## Feature Engineering Notebook
**Goal:** Transform raw columns into features that help the model learn *business signals*, not just raw values.

This is where you demonstrate maturity. Anyone can run a model on raw data.

In [27]:
# !/usr/bin/env python3
# importing libraries
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [28]:
# Load the dataset
data = pd.read_csv('../data/telco_cleaned.csv')



In [29]:
# fix data types
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
data['TotalCharges'] = data['TotalCharges'].fillna(0)

print(data.dtypes)
print(data.isnull().sum())

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
Churn_Binary          int64
num_services          int64
dtype: object
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract    

In [30]:
# --- Feature 1: Tenure Buckets ---
# Rationale: Early-stage customers behave differently from loyal ones.
# This captures non-linearity that a raw number misses.
data['tenure_bucket'] = pd.cut(
    data['tenure'],
    bins=[0, 6, 12, 24, 48, 72],
    labels=['0-6mo', '6-12mo', '1-2yr', '2-4yr', '4+yr']
)

# --- Feature 2: Average Monthly Spend vs Median ---
# Rationale: Are they a high-value customer relative to the base?
median_charge = data['MonthlyCharges'].median()
data['above_median_spend'] = (data['MonthlyCharges'] > median_charge).astype(int)

# --- Feature 3: Charges per Month of Tenure ---
# Rationale: Spend-per-tenure-month — are they getting value over time?
data['spend_per_tenure'] = data['TotalCharges'] / (data['tenure'] + 1)  # +1 to avoid div/0

# --- Feature 4: Number of Services ---
# Rationale: More services = more 'sticky'. Already computed in EDA.
service_cols = ['PhoneService', 'MultipleLines',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
data['num_services'] = (data[service_cols] == 'Yes').sum(axis=1)

# --- Feature 5: Has Security / Support Services ---
# Rationale: These are "commitment" services — likely reduce churn
data['has_security_or_support'] = ( # type: ignore
    (data['OnlineSecurity'] == 'Yes') | (data['TechSupport'] == 'Yes')
).astype(int)

# --- Feature 6: Paperless Billing Flag ---
# Rationale: Digital-native customers may behave differently
data['paperless_flag'] = (data['PaperlessBilling'] == 'Yes').astype(int)

# --- Feature 7: Electronic Check Flag ---
# Rationale: From EDA, electronic check users churn more — isolate this signal
data['electronic_check'] = (data['PaymentMethod'] == 'Electronic check').astype(int)

print('New features created.')
data[['tenure_bucket', 'above_median_spend', 'spend_per_tenure',
    'num_services', 'has_security_or_support', 'paperless_flag',
    'electronic_check']].head(10)








New features created.


,tenure_bucket,above_median_spend,spend_per_tenure,num_services,has_security_or_support,paperless_flag,electronic_check
0,0-6mo,0,14.925000,1,0,1,1
1,2-4yr,0,53.985714,3,1,0,0
2,0-6mo,0,36.050000,3,1,1,0
3,2-4yr,0,40.016304,3,1,0,0
4,0-6mo,1,50.550000,1,0,1,1
5,6-12mo,1,91.166667,5,0,1,1
6,1-2yr,1,84.756522,4,0,1,0
7,6-12mo,0,27.445455,1,1,0,0
8,2-4yr,1,105.036207,6,1,1,1
9,4+yr,0,55.364286,3,1,0,0


In [31]:
## Encode categorical variables for modeling

# Binary categoricals — map directly
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    data[col] = (data[col] == 'Yes').astype(int)

# Gender — make it 0/1 explicitly
data['gender'] = (data['gender'] == 'Male').astype(int)

# Multi-class categoricals — one-hot encode
ohe_cols = ['Contract', 'PaymentMethod', 'InternetService', 'tenure_bucket']
data = pd.get_dummies(data, columns=ohe_cols, drop_first=False)
# Drop columns that are now redundant or not useful for modeling
data.drop(['customerID', 'Churn'], axis=1, inplace=True)
# handle remaining categorical columns if any 
remaining_obj = data.select_dtypes(include='object').columns.tolist()
print('Remaining object columns:', remaining_obj)
# Handle these with pd.get_dummies() or LabelEncoder as needed

print(data.shape)
data.head()


Remaining object columns: ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
(7043, 38)


/var/folders/bh/gx67z70j0d5bn6_xhldx18jh0000gn/T/ipykernel_49782/2671395706.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  remaining_obj = data.select_dtypes(include='object').columns.tolist()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,PaymentMethod_Electronic check,PaymentMethod_Mailed check,InternetService_DSL,InternetService_Fiber optic,InternetService_No,tenure_bucket_0-6mo,tenure_bucket_6-12mo,tenure_bucket_1-2yr,tenure_bucket_2-4yr,tenure_bucket_4+yr
0,0,0,1,0,1,0,No phone service,No,Yes,No,...,True,False,True,False,False,True,False,False,False,False
1,0,0,0,0,34,1,No,Yes,No,Yes,...,False,True,True,False,False,False,False,False,True,False
2,0,0,0,0,2,1,No,Yes,Yes,No,...,False,True,True,False,False,True,False,False,False,False
3,0,0,0,0,45,0,No phone service,Yes,No,Yes,...,False,False,True,False,False,False,False,False,True,False
4,0,0,0,0,2,1,No,No,No,No,...,True,False,False,True,False,True,False,False,False,False


In [32]:
# final feature matrix check
print(data.dtypes)
# Check for any remaining nulls
print(data.isnull().sum())
# Confirm target column
print(data['Churn_Binary'].value_counts())
# all columns
print((c for c in data.columns if c not in ['Churn_Binary']))

gender                                       int64
SeniorCitizen                                int64
Partner                                      int64
Dependents                                   int64
tenure                                       int64
PhoneService                                 int64
MultipleLines                                  str
OnlineSecurity                                 str
OnlineBackup                                   str
DeviceProtection                               str
TechSupport                                    str
StreamingTV                                    str
StreamingMovies                                str
PaperlessBilling                             int64
MonthlyCharges                             float64
TotalCharges                               float64
Churn_Binary                                 int64
num_services                                 int64
above_median_spend                           int64
spend_per_tenure               

In [33]:
# Save for modeling
data.to_csv('../data/telco_features.csv', index=False)
print('Feature matrix saved.')

Feature matrix saved.
